# Winnipeg 311 Incoming Calls, 2009–2025

**Prince Kanani** · Descriptive time-series study using City of Winnipeg Open Data.

This notebook uses the archived CSV in `data/` to preserve the original 2026-02-17 source snapshot. Run from the repository root or the `notebooks/` directory.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

candidates = [Path("data/311_incoming_calls_by_day_20260217.csv"), Path("../data/311_incoming_calls_by_day_20260217.csv")]
source_file = next((p for p in candidates if p.exists()), None)
if source_file is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/ folder, with the data/ CSV present.")
print("Using archived dataset:", source_file.resolve())

## 1. Load and validate the source data

In [ ]:
# 1) Load data

df_raw = pd.read_csv(source_file, dtype=str)

df = df_raw.copy()
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

df["total_incoming_calls"] = pd.to_numeric(
    df["total_incoming_calls"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)
df["year"] = pd.to_numeric(df["year"].str.replace(",", "", regex=False), errors="coerce")
df["month_number"] = pd.to_numeric(df["month_number"].str.replace(",", "", regex=False), errors="coerce")
df["day"] = pd.to_numeric(df["day"].str.replace(",", "", regex=False), errors="coerce")

df["date"] = pd.to_datetime(
    dict(year=df["year"], month=df["month_number"], day=df["day"]),
    errors="coerce"
)

df = df.sort_values("date").reset_index(drop=True)

print("Raw range:", df["date"].min(), "to", df["date"].max())
print("Duplicate dates:", df["date"].duplicated().sum())


## 2. Complete the daily series

In [ ]:
# 2) Reindex to full daily range (adds missing day rows)

full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")

df_clean = (
    df.set_index("date")
      .reindex(full_range)
      .rename_axis("date")
      .reset_index()
)

# Derived fields
df_clean["year"] = df_clean["date"].dt.year
df_clean["month_number"] = df_clean["date"].dt.month
df_clean["day"] = df_clean["date"].dt.day
df_clean["year_month"] = df_clean["date"].dt.to_period("M").astype(str)
df_clean["is_weekend"] = df_clean["date"].dt.dayofweek >= 5

# Impute missing calls with neighbor-average (prev+next)/2 using shift function
df_clean["calls_imputed"] = df_clean["total_incoming_calls"].astype(float)
miss_mask = df_clean["calls_imputed"].isna()
df_clean.loc[miss_mask, "calls_imputed"] = (
    df_clean["calls_imputed"].shift(1) + df_clean["calls_imputed"].shift(-1)
) / 2

print("Missing days inserted:", miss_mask.sum())
print(df_clean.loc[miss_mask, ["date", "total_incoming_calls", "calls_imputed"]].head(10))


## 3. Restrict to full calendar years

In [ ]:
# 3) Final analysis window (FULL years only)
# Remove partial 2008 and partial 2026

df_final = df_clean[(df_clean["date"] >= "2009-01-01") & (df_clean["date"] <= "2025-12-31")].copy()

print("Final range:", df_final["date"].min(), "to", df_final["date"].max())
print("Years:", df_final["year"].min(), "to", df_final["year"].max())
print("Rows:", df_final.shape)


## 4. Visual style

In [ ]:
# 4) Plot colors

WARM_1 = "#C2410C"  # rust
WARM_2 = "#D97706"  # amber
WARM_3 = "#7C2D12"  # deep brown (edges/medians)
WARM_4 = "#92400E"  # warm brown

## 5. Yearly incoming calls

In [ ]:
# Figure 1 

year_totals = df_final.groupby("year")["calls_imputed"].sum().reset_index(name="total_calls")

plt.figure(figsize=(9.0, 4.8))
plt.bar(year_totals["year"], year_totals["total_calls"], color=WARM_2, edgecolor=WARM_3)
plt.title("Total 311 Incoming Calls by Year (2009–2025)")
plt.xlabel("Year")
plt.ylabel("Total calls")



plt.tight_layout()
plt.show()


## 6. Recent monthly totals

In [ ]:
# Figure 2

df_232425 = df_final[df_final["year"].isin([2023, 2024, 2025])].copy()

monthly_totals = (
    df_232425.groupby(["year", "month_number"])["calls_imputed"]
    .sum()
    .reset_index(name="total_calls")
)

plt.figure(figsize=(9.0, 4.8))
color_map = {2023: WARM_4, 2024: WARM_2, 2025: WARM_1}

for y in [2023, 2024, 2025]:
    temp = monthly_totals[monthly_totals["year"] == y].sort_values("month_number")
    plt.plot(temp["month_number"], temp["total_calls"], marker="o", label=str(y), color=color_map[y])

plt.title("Monthly Total 311 Calls: 2023 vs 2024 vs 2025")
plt.xlabel("Month")
plt.ylabel("Total calls in month")
plt.xticks(range(1, 13))
plt.legend()



plt.tight_layout()
plt.show()


## 7. Monthly average daily volume

In [ ]:
# Figure 3 
df_2425 = df_final[df_final["year"].isin([2024, 2025])].copy()

monthly_avg = (
    df_2425.groupby(["year", "month_number"])["calls_imputed"]
    .mean()
    .reset_index(name="avg_daily_calls")
)

wide_2425 = monthly_avg.pivot(index="month_number", columns="year", values="avg_daily_calls").reset_index()

plt.figure(figsize=(9.0, 4.8))
plt.plot(wide_2425["month_number"], wide_2425[2024], marker="o", label="2024", color=WARM_2)
plt.plot(wide_2425["month_number"], wide_2425[2025], marker="o", label="2025", color=WARM_1)

plt.title("Monthly Average Daily 311 Calls: 2024 vs 2025")
plt.xlabel("Month")
plt.ylabel("Average daily calls")
plt.xticks(range(1, 13))
plt.legend()


plt.tight_layout()
plt.show()


## 8. Weekday and weekend demand

In [ ]:
# Figure 4 
ww_monthly = (
    df_232425.groupby(["year", "month_number", "is_weekend"])["calls_imputed"]
    .mean()
    .reset_index(name="avg_daily_calls")
)

# convert to a month-start datetime for a clean time axis
ww_monthly["ym_date"] = pd.to_datetime(dict(year=ww_monthly["year"], month=ww_monthly["month_number"], day=1))
ww_monthly = ww_monthly[ww_monthly["ym_date"] <= pd.Timestamp("2025-12-01")]

weekday_line = ww_monthly[ww_monthly["is_weekend"] == False].sort_values("ym_date")
weekend_line = ww_monthly[ww_monthly["is_weekend"] == True].sort_values("ym_date")

plt.figure(figsize=(9.0, 4.8))
plt.plot(weekday_line["ym_date"], weekday_line["avg_daily_calls"], marker="o", label="Weekday", color=WARM_1)
plt.plot(weekend_line["ym_date"], weekend_line["avg_daily_calls"], marker="o", label="Weekend", color=WARM_2)

plt.title("Average Daily 311 Calls (Monthly): Weekday vs Weekend (2023–2025)")
plt.xlabel("Year-Month")
plt.ylabel("Average daily calls")
plt.legend()

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.xticks(rotation=45, ha="right")


plt.tight_layout()
plt.show()


## 9. Long-run rolling averages

In [ ]:
# Figure 5 
ts = df_final.set_index("date")["calls_imputed"].sort_index()
roll30 = ts.rolling(30, min_periods=15).mean()
roll90 = ts.rolling(90, min_periods=45).mean()

plt.figure(figsize=(9.0, 4.8))
plt.plot(roll30.index, roll30.values, label="30-day avg", color=WARM_2)
plt.plot(roll90.index, roll90.values, label="90-day avg (long-term trend)", color=WARM_1)

plt.axvspan(pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-31"), alpha=0.18, color=WARM_2)

# Add space (padding) around graph
start = pd.Timestamp("2009-01-01") - pd.DateOffset(months=6)
end   = pd.Timestamp("2025-12-31") + pd.DateOffset(months=6)

ax = plt.gca()
ax.set_xlim(start, end)

plt.title("Winnipeg 311 Calls Trend (Rolling Averages), 2009–2025")
plt.xlabel("Date")
plt.ylabel("Calls")
plt.legend()

plt.tight_layout()

plt.tight_layout()
plt.show()


## 10. Daily distributions

In [ ]:
# Figure 6
weekday_vals = df_final.loc[~df_final["is_weekend"], "calls_imputed"].dropna().values
weekend_vals = df_final.loc[df_final["is_weekend"], "calls_imputed"].dropna().values

plt.figure(figsize=(7.8, 5.0))
bp = plt.boxplot([weekday_vals, weekend_vals], tick_labels=["Weekday", "Weekend"], patch_artist=True)

bp["boxes"][0].set(facecolor=WARM_2, edgecolor=WARM_3)
bp["boxes"][1].set(facecolor=WARM_1, edgecolor=WARM_3)

for median in bp["medians"]:
    median.set(color=WARM_3, linewidth=1.5)

plt.title("Weekday vs Weekend Distribution of Daily 311 Calls (2009–2025)")
plt.ylabel("Daily calls")



plt.tight_layout()
plt.show()


## 11. Highest and lowest years

In [ ]:
# Highest / Lowest year totals

year_totals = (
    df_final.groupby("year")["calls_imputed"]
    .sum()
    .reset_index(name="total_calls")
)

highest_year = year_totals.loc[year_totals["total_calls"].idxmax()]
lowest_year  = year_totals.loc[year_totals["total_calls"].idxmin()]

print("Highest full year:", int(highest_year["year"]), f"~{highest_year['total_calls']:,.0f}")
print("Lowest full year:", int(lowest_year["year"]),  f"~{lowest_year['total_calls']:,.0f}")


## 12. Yearly summary table

In [ ]:
display(year_totals)

## Interpretation and limitations

The notebook reports a **descriptive** change in call volumes, not a controlled causal effect. The 2024–2025 comparison has no adjustment for changes in reporting systems, caller behaviour, weather, population, or municipal operations. The dataset contains daily totals only; it cannot identify why calls were made. The single missing day in the original source is imputed using the two adjacent days; this should be considered when reproducing annual totals.

**Data:** City of Winnipeg, [311 Incoming Calls by Day](https://data.winnipeg.ca/dataset/311-Incoming-Calls-By-Day/p3xc-y6xi), licensed under the [Open Government Licence – Winnipeg](https://data.winnipeg.ca/stories/s/Open-Data-Licence/rwzh-4ijx/).